# 1. Conda environment setup (Windows edition)

Language: **English** | [中文](../cn/01_environment_setup.ipynb)

This notebook creates or updates the repository's `cofkit` Conda environment and registers it as a Jupyter kernel. This native-Windows edition uses regular Python cells and requires neither PowerShell nor WSL. Run it from a clone of the repository.

Commented cells show the corresponding direct Python checks where applicable.

## Tutorial series

1. **Environment setup** (this notebook)
2. [First COF build](02_first_cof_build.ipynb)
3. [Topology, connectivity, and linkage examples](03_topologies_connectivities_and_linkages.ipynb)
4. [Pore analysis with Zeo++](04_zeopp_pore_analysis.ipynb)

Launch Jupyter from the repository root so that all relative paths resolve consistently. Regular Python cells call Conda through `subprocess`, so no PowerShell shell state needs to persist across cells.

## Inspect the checked-in environment

The first line of `environment.yml` fixes the environment name as `cofkit`. The same name is used by this entire tutorial series.

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
environment_file = repo_root / "environment.yml"
if not environment_file.is_file():
    raise FileNotFoundError(
        "Launch Jupyter from the cofkit repository root, then rerun this cell."
    )

print(f"Repository root: {repo_root}\n")
print("\n".join(environment_file.read_text(encoding="utf-8").splitlines()[:24]))

In [ ]:
# Python equivalent (commented out): inspect the environment name.
# from pathlib import Path
# environment_name = next(
#     line.split(":", 1)[1].strip()
#     for line in Path("environment.yml").read_text().splitlines()
#     if line.startswith("name:")
# )
# print(environment_name)

## Create or update the environment

This cell first checks whether the `cofkit` CLI is already available on `PATH`. If it is, the cell exits successfully without changing the environment—useful when choosing **Run All**. Otherwise, it creates `cofkit` when absent or updates it from `environment.yml`, installs the local package without replacing Conda-resolved dependencies, and registers a `Python (cofkit)` kernel for Notebooks 2–4.

> Environment resolution and the Open Babel installation can take several minutes on a first run.

In [ ]:
import json
import os
from pathlib import Path
import shutil
import subprocess

cofkit_command = shutil.which("cofkit")
if cofkit_command:
    print(
        f"cofkit CLI is already available at {cofkit_command}; "
        "skipping environment and package setup."
    )
else:
    repo_root = Path.cwd().resolve()
    environment_file = repo_root / "environment.yml"
    if not environment_file.is_file():
        raise FileNotFoundError(
            "Launch Jupyter from the cofkit repository root, then rerun this cell."
        )

    conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
    if not conda:
        raise RuntimeError(
            "Conda was not found. Launch Jupyter from Miniforge, Miniconda, "
            "or Anaconda and rerun this cell."
        )

    environment_info = json.loads(
        subprocess.run(
            [conda, "env", "list", "--json"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout
    )
    environment_exists = any(
        Path(path).name.casefold() == "cofkit"
        for path in environment_info["envs"]
    )
    if environment_exists:
        environment_command = [
            conda, "env", "update", "--name", "cofkit",
            "--file", str(environment_file), "--prune",
        ]
    else:
        environment_command = [
            conda, "env", "create", "--file", str(environment_file),
        ]
    subprocess.run(environment_command, cwd=repo_root, check=True)
    subprocess.run(
        [conda, "run", "-n", "cofkit", "python", "-m", "pip",
         "install", "--no-deps", "."],
        cwd=repo_root,
        check=True,
    )
    subprocess.run(
        [conda, "install", "--name", "cofkit", "--channel", "conda-forge",
         "--yes", "ipykernel"],
        cwd=repo_root,
        check=True,
    )
    subprocess.run(
        [conda, "run", "-n", "cofkit", "python", "-m", "ipykernel",
         "install", "--user", "--name", "cofkit",
         "--display-name", "Python (cofkit)"],
        cwd=repo_root,
        check=True,
    )
    print("The cofkit environment and Jupyter kernel are ready.")

In [ ]:
# Conda does not expose a stable public Python API for environment creation.
# The runnable cell above therefore calls Conda and pip safely through subprocess arguments.

If this notebook was opened with another kernel, you may now select **Python (cofkit)** from Jupyter's kernel menu. The remaining executable cells explicitly use `conda run -n cofkit`, so switching is optional here. Notebooks 2–4 declare `cofkit` as their default kernel.

## Verify the installation

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

repo_root = Path.cwd().resolve()
conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
if not conda:
    raise RuntimeError("Conda was not found. Run the environment setup cell first.")

commands = [
    [conda, "run", "-n", "cofkit", "cofkit", "--version"],
    [conda, "run", "-n", "cofkit", "python", "-c",
     "import cofkit; print('Python import:', cofkit.__version__)"],
    [conda, "run", "-n", "cofkit", "cofkit", "build", "list-templates"],
]
for command in commands:
    subprocess.run(command, cwd=repo_root, check=True)

In [ ]:
# Python equivalent (commented out): verify that the package imports.
# import cofkit
# print(cofkit.__version__)
#
# from cofkit import ReactionLibrary
# library = ReactionLibrary.builtin()
# print(library)

## Next step

The environment is ready. Continue with [Notebook 2](02_first_cof_build.ipynb) to build the first COF.